In [1]:
import os
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier, plot_importance
import seaborn as sns
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
import shap
import numpy as np
from collections import defaultdict
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import accuracy_score

# must be set so that figures do not get cut off
plt.rcParams['figure.autolayout'] = True

In [2]:
# import drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
region_name = 'eastern_east_africa'
season = 'MAM'
df = pd.read_csv('/content/drive/My Drive/capstone_data/ml_EEA_MAM_precip_predictors/eastern_east_africa_MAM_ml_data_monthly.csv')
df = df[[col for col in df.columns if not (col.endswith('L1') or col.endswith('L2'))]] # drop L1 and L2
df['decade'] = (df['year'] // 10) * 10 # add column decade

In [4]:
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

In [5]:
if 'precip' in df.columns:
    df.drop(columns=['precip'], inplace=True)
if 'region' in df.columns:
    df.drop(columns=['region'], inplace=True)

In [6]:
df

,year,month,tercile,nino_34_L3,nino_4_L3,western_west_v_L3,northern_west_v_L3,southern_west_v_L3,SWIO_L3,IOD_west_L3,...,nino_4_L8,western_west_v_L8,northern_west_v_L8,southern_west_v_L8,SWIO_L8,IOD_west_L8,IOD_east_L8,SA_precip,EEA_precip,decade
0,1982,3,n,-0.147724,-0.043525,-0.699381,-0.265528,-0.038946,0.072711,-1.124293,...,-1.235995,0.180759,-0.349686,-0.977762,-0.379167,-0.869233,0.157105,-0.919264,0.355706,1980
1,1982,4,an,-0.304030,-0.096568,-0.718615,-1.263759,-1.089806,0.645471,-0.695839,...,-1.108068,-0.361421,-1.399467,-1.121600,-0.584303,-0.886227,0.219707,0.095751,-0.547513,1980
2,1982,5,an,-0.637298,-0.166301,-0.843808,-1.275156,-0.524595,0.961337,-0.471048,...,-0.536004,-0.397634,-0.604492,-0.681802,-1.069736,-1.380038,0.197191,-0.266583,-0.823349,1980
3,1983,3,bn,1.832082,0.678931,-1.260516,-1.987008,-2.184105,-0.892840,0.328844,...,0.391133,-2.249874,-1.411861,-1.020552,-0.084088,-0.180941,-1.284622,-0.500735,2.578269,1980
4,1983,4,bn,0.122836,0.027884,-0.858548,-0.520048,0.469384,0.975110,-0.717409,...,0.275922,-2.186668,-1.189379,-1.421364,0.079620,0.010555,-1.113700,-0.389758,0.235423,1980
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,2023,4,an,-0.804207,-0.398429,0.850032,0.986100,1.506256,-0.088025,0.759078,...,-1.770854,1.645279,1.506039,2.261708,0.628908,-0.852394,1.743456,0.512960,-0.483339,2020
125,2023,5,bn,-0.894453,-0.432452,0.868748,1.015477,1.502159,0.341280,-0.126079,...,-1.778606,2.000044,1.236147,2.317537,0.562597,-0.816236,1.259682,0.409825,-0.825349,2020
126,2024,3,n,1.693982,1.841095,0.813586,1.798863,0.210130,1.448626,2.988746,...,1.475082,1.053588,0.962178,1.125513,2.306043,1.217969,0.388742,-0.728363,2.516727,2020
127,2024,4,an,-0.628054,-0.758568,1.146019,0.844711,0.569867,2.413360,-0.335014,...,1.762354,0.601438,0.823420,0.725047,2.307410,1.959505,0.050726,-0.873584,2.610538,2020


In [7]:
# Separate train/test by year
train_years = list(range(1981, 2016))
test_years = list(range(2016, 2025))
train_df = df[df['year'].isin(train_years)]
test_df = df[df['year'].isin(test_years)]

# Label encode targets
le = LabelEncoder()
y_train_enc = le.fit_transform(train_df['tercile'])
y_test_enc = le.transform(test_df['tercile'])

X_train = train_df.drop(columns=['tercile', 'decade'])
X_test = test_df.drop(columns=['tercile', 'decade'])
groups = train_df['year']
#decades = train_df['decade']

unique_years = train_df['year'].unique()
#year_to_decade = {year: (year // 10) * 10 for year in unique_years}

# Build decade-balanced folds
n_splits = 5
folds = [[] for _ in range(n_splits)]
years_by_decade = train_df.groupby('decade')['year'].unique()

for decade, years in years_by_decade.items():
    years = list(years)
    print(years)
    np.random.shuffle(years)
    for i, year in enumerate(years):
        folds[i % n_splits].append(year)

# Cross-Validation
best_model = None
best_score = 0

for i in range(n_splits):
    val_years = folds[i]
    train_years_cv = [y for j in range(n_splits) if j != i for y in folds[j]]

    train_idx = train_df['year'].isin(train_years_cv)
    val_idx = train_df['year'].isin(val_years)

    X_tr, y_tr = X_train[train_idx], y_train_enc[train_idx]
    X_val, y_val = X_train[val_idx], y_train_enc[val_idx]

    print(f"\nFold {i + 1}:")
    print(f"  Train years: {sorted(train_years_cv)}")
    print(f"  Validation years: {sorted(val_years)}")
    print(f"  Train size: {X_tr.shape[0]}, Validation size: {X_val.shape[0]}")

    model = XGBClassifier(eval_metric='mlogloss')
    model.fit(X_tr, y_tr)
    val_preds = model.predict(X_val)
    acc = accuracy_score(y_val, val_preds)

    print(f"Fold {i+1} - Val Accuracy: {acc:.4f}")

    if acc > best_score:
        best_score = acc
        best_model = model

[np.int64(1982), np.int64(1983), np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989)]
[np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999)]
[np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009)]
[np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015)]

Fold 1:
  Train years: [np.int64(1983), np.int64(1984), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1999), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013)]
  Validatio

In [11]:
y_pred = best_model.predict(X_test)

# Predict probabilities for each class
y_proba = best_model.predict_proba(X_test)

s = 'cv_precip'
# Save the model
model_save_path = f"/content/drive/My Drive/Capstone_data/EEA_MAM_seasonal_model_{s}.json"
#model_save_path = os.path.join(model_folder_seasonal, f"{region_name}_{season}_seasonal_model.json")
best_model.save_model(model_save_path)

# Save the classification report as a text file
classification_report_path = f"/content/drive/My Drive/Capstone_data/EEA_MAM_seasonal_model_classification_report_{s}.json"
with open(classification_report_path, 'w') as f:
    f.write(f"{region_name} {season} Long Lead Seasonal Model\n=== Classification Report ===\n")
    f.write(classification_report(
        y_test_enc,
        y_pred,
        labels=[0, 1, 2],  # Ensure all label indices are present
        target_names=le.classes_,
        zero_division=0  # Handle division by zero (i.e N was not in test data)
    ))

# Confusion matrix
conf_mat = confusion_matrix(y_test_enc, y_pred, labels=[0, 1, 2])
plt.figure(figsize=(8, 6))
sns.heatmap(conf_mat, annot=True, fmt='d',
            xticklabels=le.classes_, yticklabels=le.classes_,
            cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'{region_name} {season} \nLong Lead Seasonal Model\nConfusion Matrix')

# Save the confusion matrix
conf_matrix_file = f'/content/drive/My Drive/Capstone_data/EEA_MAM_seasonal_model_confusion_matrix_{s}.png'
plt.savefig(conf_matrix_file, dpi=300)
plt.close()

# Feature importance plot
plt.figure(figsize=(20, 12))
plot_importance(model, max_num_features=20, importance_type='weight', height=0.5)
plt.title(f'{region_name} {season} \nLong Lead Seasonal Model \nTop 20 Feature Importances')

# Save the feature importance plot
feature_importance_file = f'/content/drive/My Drive/Capstone_data/EEA_MAM_seasonal_model_feature_importance_{s}.png'
plt.savefig(feature_importance_file, dpi=300)
plt.close()

##### Create explainer
#X_train_encoded = X_train_encoded.astype(float)
#X_test_encoded = X_test_encoded.astype(float)

explainer = shap.Explainer(model, X_train)

# Compute SHAP values
shap_values = explainer(X_test)

# get the class names from the model's encoder
class_names = le.classes_

# Use matplotlib-compatible SHAP summary bar plot
plt.figure(figsize=(20, 12))
shap.summary_plot(shap_values.values, X_test, plot_type="bar", class_names=class_names, show=False) # use class names list to label shap plot
plt.title(f'{region_name} {season} \nLong Lead Seasonal Model \nTop 20 SHAP Feature Importances')
plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust layout to fit title

shap_feature_importance_file = f'/content/drive/My Drive/Capstone_data/EEA_MAM_seasonal_model_shap_feature_importance_{s}.png'
plt.savefig(shap_feature_importance_file, dpi=300)
plt.close()

# Probabilistic forecast details

# Reset index to ensure alignment
test_info = test_df[['month','year']].reset_index(drop=True)  # Only 'year' for seasonal data
proba_df = pd.DataFrame(y_proba, columns=[f"proba_{cls}" for cls in le.classes_])

# Concatenate year and predicted probabilities
results_df = pd.concat([test_info, proba_df], axis=1)

# Add predicted and true class labels
results_df['true_label'] = le.inverse_transform(y_test_enc)
results_df['predicted_label'] = le.inverse_transform(y_pred)
results_df = results_df.sort_values(by=['month', 'year']).reset_index(drop=True)

# Save the results
forecast_details_file = f"/content/drive/My Drive/Capstone_data/EEA_MAM_seasonal_model_probabilistic_forecast_details_{s}.txt"
with open(forecast_details_file, 'w') as f:
    f.write(f"{region_name} {season} Long Lead Seasonal Model\n=== Seasonal Predictions with Probabilities ===\n")
    f.write(results_df.to_string(index=False))